In [1]:
import os
import random
from pathlib import Path
from PIL import Image
import numpy as np

# ===== CONFIG =====
RAW_DIR = Path(r"D:\Micro Imagine cup\NeuroAdaptive\data\dyslexia_handwriting\raw")
OUT_DIR = Path(r"D:\Micro Imagine cup\NeuroAdaptive\data\dyslexia_handwriting\processed")
IMG_SIZE = (224, 224)
VAL_SPLIT = 0.2
SEED = 42

random.seed(SEED)

# Create output folders
for split in ["train", "val"]:
    for label in ["Dyslexia_Positive", "Dyslexia_Negative"]:
        (OUT_DIR / split / label).mkdir(parents=True, exist_ok=True)

# Collect images
image_paths = []
labels = []

for label_folder in ["Dyslexia_Positive", "Dyslexia_Negative"]:
    folder = RAW_DIR / label_folder
    for img in folder.glob("*"):
        if img.suffix.lower() in [".png", ".jpg", ".jpeg"]:
            image_paths.append(img)
            labels.append(label_folder)

# Shuffle dataset while keeping image–label mapping intact
combined = list(zip(image_paths, labels))
random.shuffle(combined)
image_paths, labels = zip(*combined)

# Split into train/val
val_count = int(len(image_paths) * VAL_SPLIT)

train_imgs = image_paths[val_count:]
train_labels = labels[val_count:]

val_imgs = image_paths[:val_count]
val_labels = labels[:val_count]

def preprocess_and_save(img_list, label_list, split):
    for img_path, label in zip(img_list, label_list):
        try:
            # Load
            img = Image.open(img_path).convert("RGB")
            # Resize
            img = img.resize(IMG_SIZE)
            # Normalize to 0–1
            img_arr = np.array(img) / 255.0
            # Convert back to image for saving
            img_out = Image.fromarray((img_arr * 255).astype(np.uint8))
            # Save to correct folder
            save_path = OUT_DIR / split / label / img_path.name
            img_out.save(save_path)
        except Exception as e:
            print(f"❌ Error processing {img_path.name}: {e}")

# Run preprocessing and save images
print("🚀 Preprocessing training images...")
preprocess_and_save(train_imgs, train_labels, "train")

print("🚀 Preprocessing validation images...")
preprocess_and_save(val_imgs, val_labels, "val")

print(f"\n✅ Done! {len(train_imgs)} train and {len(val_imgs)} val images saved in:")
print(OUT_DIR)


🚀 Preprocessing training images...
🚀 Preprocessing validation images...

✅ Done! 200 train and 49 val images saved in:
D:\Micro Imagine cup\NeuroAdaptive\data\dyslexia_handwriting\processed
